In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from textwrap import wrap

clean_df = pd.read_csv("../data/clean-survey.csv")
dummy_df = pd.read_csv("../data/dummy-data.csv")

def wrap_labels(labels, width=18):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]

def pct(mask):
    return mask.mean() * 100
df_problem = clean_df.copy()
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
for col in ["info_visibility_diff", "feedback_not_useful_frequency", "stop_checking_data_frequency"]:
    df_problem[f"{col}_num"] = df_problem[col].map(frequency_scale)

main_blue = "#3d7ea6"
contrast_red = "#d1495b"


### Step 3: Understanding the Problem Space

This step validates whether the wearable or health-tracking problem space is relevant for the survey participants. It uses Likert-scale problem indicators and frequency-based friction indicators, matching the guide's focus on relevance, frequency, and severity.

#### 1. Problem Indicators

For Likert items, higher values indicate stronger agreement with the problem statement. For frequency items, `Rarely` to `Always` is mapped to an ordered `1-4` scale.

In [ ]:
likert_items = {
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Frustration": "frustrated_feeling",
    "Battery Anxiety": "battery_anxiety",
    "Actionable Insight Gap": "actionable_insight_gap",
}
frequency_items = {
    "Visibility Difficulties": "info_visibility_diff_num",
    "Feedback Not Useful": "feedback_not_useful_frequency_num",
    "Stopped Checking Data": "stop_checking_data_frequency_num",
}

likert_df = df_problem[list(likert_items.values())].rename(columns={v: k for k, v in likert_items.items()})
freq_df = df_problem[list(frequency_items.values())].rename(columns={v: k for k, v in frequency_items.items()})

likert_summary = likert_df.agg(["mean", "median", "std"]).T.round(2)
likert_summary["high n"] = likert_df.ge(3).sum()
likert_summary["high %"] = likert_df.ge(3).mean().mul(100).round(1)

freq_summary = freq_df.agg(["mean", "median", "std"]).T.round(2)
freq_summary["sometimes+ n"] = freq_df.ge(2).sum()
freq_summary["sometimes+ %"] = freq_df.ge(2).mean().mul(100).round(1)
freq_summary["often+ %"] = freq_df.ge(3).mean().mul(100).round(1)

display(likert_summary.sort_values("high %", ascending=False))
display(freq_summary.sort_values("sometimes+ %", ascending=False))

evidence = pd.concat([
    likert_summary["high %"],
    freq_summary["sometimes+ %"],
]).sort_values()

fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
ax.barh(range(len(evidence)), evidence.values, color=main_blue)
ax.set_yticks(range(len(evidence)))
ax.set_yticklabels(wrap_labels(evidence.index, width=26))
ax.set_xlim(0, 100)
ax.set_xlabel("Participants (%)")
ax.set_title("Problem-Space Evidence Summary")
ax.bar_label(ax.containers[0], labels=[f"{v:.1f}%" for v in evidence.values], padding=3)
plt.show()

The problem evidence is moderate but recurring. Among agreement-based items, **battery anxiety** and **actionable insight gap** are the clearest concerns, each with `44.4%` high agreement. **Information overload** is also present, with `33.3%` high agreement.

The frequency-based items show repeated friction rather than isolated problems. Visibility difficulties, feedback not being useful, and stopping checking data are each reported sometimes or more by `77.8%` of participants.

#### 2. Current Users Compared with Former or Non-Current Users

This comparison checks whether problem indicators differ between current users and participants who do not currently use wearable technologies. It is descriptive only because the subgroups are small.

In [ ]:
df_problem["user_status"] = df_problem["usage_duration"].apply(
    lambda value: "Former / non-current users" if value == "I do not currently use them" else "Current users"
)

comparison_items = {
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Frustration": "frustrated_feeling",
    "Battery Anxiety": "battery_anxiety",
    "Actionable Insight Gap": "actionable_insight_gap",
    "Visibility Difficulties": "info_visibility_diff_num",
    "Feedback Not Useful": "feedback_not_useful_frequency_num",
    "Stopped Checking Data": "stop_checking_data_frequency_num",
}

group_comparison = (
    df_problem.groupby("user_status")[list(comparison_items.values())]
    .mean()
    .T.rename(index={v: k for k, v in comparison_items.items()})
    .round(2)
)
display(group_comparison)

fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
group_comparison.plot(kind="bar", ax=ax, color=[main_blue, contrast_red])
ax.set_title("Problem Indicators by Current Use Status")
ax.set_ylabel("Average response")
ax.set_xlabel("")
ax.set_xticklabels(wrap_labels(group_comparison.index, width=18), rotation=0)
ax.legend(title="Participant group", loc="upper right")
plt.show()

Former or non-current users report higher averages for several frictions, including interpretation difficulty, frustration, battery anxiety, visibility difficulties, and feedback not being useful. This supports the idea that effortful or insufficiently rewarding experiences may be connected to disengagement.

However, current users report a higher average for stopping checking data (`2.54`) than former or non-current users (`2.00`). This suggests that disengagement can occur within continued use, not only after people stop using the technology entirely.

#### Step 3 Reflection

The problem space appears relevant for this sample. The strongest pattern is not extreme severity across every item, but repeated medium-level friction around actionability, feedback usefulness, visibility, battery anxiety, and ongoing data checking. For design, this points toward reducing cognitive and practical effort while making feedback easier to act on.